In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from glob import glob
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import KFold


In [ ]:

# ==========================================
# 2. KONFIGURASI (HYPERPARAMETERS)
# ==========================================
class Config:
    SEED = 42
    IMG_SIZE = 256        # Ukuran input (bisa dinaikkan ke 384/512 jika GPU kuat)
    BATCH_SIZE = 16       # Sesuaikan dengan RAM GPU
    EPOCHS = 25           # Jumlah epoch per fold
    LR = 2e-4             # Learning rate
    N_FOLDS = 5           # Jumlah lipatan cross-validation
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS = 2
    
    # Path (Sesuaikan jika path berbeda di environmentmu)
    TRAIN_IMG_DIR = "/content/drive/MyDrive/ARA-ITS/dataset/train/images"
    TRAIN_MASK_DIR = "/content/drive/MyDrive/ARA-ITS/dataset/train/mask"
    TEST_IMG_DIR = "/content/drive/MyDrive/ARA-ITS/dataset/test/images"

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(Config.SEED)

In [ ]:
# ==========================================
# 3. UTILITIES (RLE & METRIC)
# ==========================================
# Fungsi encode dari panitia
def encode_rle(mask: np.ndarray, pos_value: int = 255) -> str:
    binary = (mask == pos_value).astype(np.uint8)
    pixels = binary.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)

# Metrik Dice untuk monitoring saat training
def dice_coef(y_pred, y_true, smooth=1e-7):
    y_pred = y_pred.sigmoid()
    y_pred = (y_pred > 0.5).float()
    
    intersection = (y_pred * y_true).sum()
    return (2. * intersection + smooth) / (y_pred.sum() + y_true.sum() + smooth)

In [ ]:
# ==========================================
# 4. DATASET & AUGMENTATION
# ==========================================
class PotholeDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, transform=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load Image
        img_path = self.img_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.mask_paths:
            # Load Mask (Training)
            mask_path = self.mask_paths[idx]
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            # Pastikan mask biner (0 atau 1) untuk training
            mask = np.where(mask > 127, 1.0, 0.0)
            
            if self.transform:
                augmented = self.transform(image=image, mask=mask)
                image = augmented['image']
                mask = augmented['mask']
                
            # Bentuk mask harus [1, H, W] agar cocok dengan output model
            mask = mask.unsqueeze(0)
            return image, mask
        else:
            # Test set (tanpa mask)
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, os.path.basename(img_path)

In [ ]:
# Augmentasi "Advanced": Membuat model tahan terhadap variasi warna dan posisi
def get_transforms(data):
    if data == 'train':
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=30, p=0.5),
            A.OneOf([
                A.RandomBrightnessContrast(p=0.5),
                A.HueSaturationValue(p=0.5),
            ], p=0.3),
            A.Normalize(
                mean=[0.485, 0.456, 0.406], # Mean standard ImageNet
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ])
    elif data == 'valid':
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ])


In [ ]:
# ==========================================
# 5. LOSS FUNCTION (Dice + BCE)
# ==========================================
# Kombinasi ini membantu model belajar struktur global (Dice) dan pixel akurasi (BCE)
class BCEDiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(BCEDiceLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1):
        # BCE Loss
        bce_loss = self.bce(inputs, targets)
        
        # Dice Loss
        inputs = torch.sigmoid(inputs)
        
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)
        
        return 0.5 * bce_loss + 0.5 * dice_loss


In [ ]:
# ==========================================
# 6. TRAINING HELPER FUNCTIONS
# ==========================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    
    for images, masks in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        masks = masks.to(device).float()
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    return running_loss / len(loader)

def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    val_dice = 0.0
    
    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Valid", leave=False):
            images = images.to(device)
            masks = masks.to(device).float()
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            running_loss += loss.item()
            val_dice += dice_coef(outputs, masks).item()
            
    return running_loss / len(loader), val_dice / len(loader)

In [ ]:
# ==========================================
# 7. MAIN: K-FOLD TRAINING LOOP
# ==========================================
img_paths = np.array(sorted(glob(f"{Config.TRAIN_IMG_DIR}/*")))
mask_paths = np.array(sorted(glob(f"{Config.TRAIN_MASK_DIR}/*")))

kf = KFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)

print(f"Mulai Training dengan {Config.N_FOLDS}-Fold Cross Validation...")

# Menyimpan path model terbaik dari setiap fold
best_models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(img_paths)):
    print(f"\n{'='*20} Fold {fold+1}/{Config.N_FOLDS} {'='*20}")
    
    # Split Data
    X_train, X_val = img_paths[train_idx], img_paths[val_idx]
    y_train, y_val = mask_paths[train_idx], mask_paths[val_idx]
    
    # Dataset & Loader
    train_ds = PotholeDataset(X_train, y_train, transform=get_transforms('train'))
    val_ds = PotholeDataset(X_val, y_val, transform=get_transforms('valid'))
    
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    
    # Model: U-Net dengan Backbone ResNet34 (Transfer Learning)
    # Ini kunci kemenangan: pretrained encoder mengenali tekstur jalan dengan sangat baik
    model = smp.Unet(
        encoder_name="resnet34",        
        encoder_weights="imagenet",     
        in_channels=3,                  
        classes=1,                      
    ).to(Config.DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)
    criterion = BCEDiceLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)
    
    best_dice = 0.0
    best_model_path = f"unet_resnet34_fold{fold}.pth"
    
    for epoch in range(Config.EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, Config.DEVICE)
        val_loss, val_dice = valid_one_epoch(model, val_loader, criterion, Config.DEVICE)
        
        scheduler.step()
        
        # Save model jika performa membaik
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), best_model_path)
            print(f"Epoch {epoch+1}: Val Dice improved to {val_dice:.4f} | Loss: {train_loss:.4f}")
        else:
            print(f"Epoch {epoch+1}: Val Dice {val_dice:.4f} | Loss: {train_loss:.4f}")
            
    best_models.append(best_model_path)
    
    # Bersihkan memori GPU
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()

print("\nTraining Selesai! Model terbaik tersimpan.")

In [ ]:
# ==========================================
# 8. INFERENCE & ENSEMBLE (PREDIKSI)
# ==========================================
# Load Test Data
test_paths = sorted(glob(f"{Config.TEST_IMG_DIR}/*"))
test_ds = PotholeDataset(test_paths, transform=get_transforms('valid'))
test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

# Load semua model dari 5 fold
models = []
for path in best_models:
    model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1)
    model.load_state_dict(torch.load(path))
    model.to(Config.DEVICE)
    model.eval()
    models.append(model)

print("Memulai Prediksi Ensemble...")

submission_data = []

# Post-processing helper: Hapus objek kecil (noise)
def remove_small_blobs(mask, min_size=50):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    new_mask = np.zeros_like(mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            new_mask[labels == i] = 255
    return new_mask

with torch.no_grad():
    for images, ids in tqdm(test_loader):
        images = images.to(Config.DEVICE)
        
        # Ensemble Prediction: Rata-rata dari 5 model
        avg_preds = torch.zeros((images.size(0), 1, Config.IMG_SIZE, Config.IMG_SIZE)).to(Config.DEVICE)
        
        for model in models:
            preds = model(images)
            preds = torch.sigmoid(preds)
            avg_preds += preds
            
        avg_preds /= len(models) # Rata-rata
        
        # Pindahkan ke CPU untuk processing numpy
        avg_preds = avg_preds.cpu().numpy()
        
        for i in range(len(ids)):
            image_id = ids[i]
            pred_mask = avg_preds[i, 0]
            
            # 1. Resize kembali ke ukuran asli jika perlu (disini asumsi panitia menerima size berapapun)
            # Namun biasanya submission diminta ukuran asli.
            # Kita akan resize ke ukuran asli gambar test jika diketahui, 
            # tapi untuk aman kita anggap resize saat encode atau tetap di 256 jika diizinkan.
            # *Pro Tip*: Sebaiknya load gambar asli untuk tahu ukurannya, lalu resize pred_mask.
            original_img = cv2.imread(os.path.join(Config.TEST_IMG_DIR, image_id))
            h, w = original_img.shape[:2]
            pred_mask = cv2.resize(pred_mask, (w, h))
            
            # 2. Thresholding (Adaptive atau Hard)
            # Kita gunakan hard threshold 0.5 yang aman
            binary_mask = (pred_mask > 0.5).astype(np.uint8) * 255
            
            # 3. Post-Processing: Remove noise
            clean_mask = remove_small_blobs(binary_mask, min_size=100)
            
            # 4. RLE Encoding
            # Jika mask kosong, hasilkan string kosong (handled by function logic or check)
            if clean_mask.sum() == 0:
                rle_str = ""
            else:
                rle_str = encode_rle(clean_mask)
            
            submission_data.append([image_id, rle_str])


In [ ]:
# ==========================================
# 9. SUBMISSION
# ==========================================
df_sub = pd.DataFrame(submission_data, columns=['ImageId', 'rle'])
df_sub.to_csv('submission.csv', index=False)

print("Submission.csv berhasil dibuat!")
df_sub.head()